# NB-05: Figure & Image Dependency Checker

Scans all `\\includegraphics` calls in the HypatiaX paper bundle, detects `\\fbox`
placeholders, checks whether required image files exist on disk, and audits every
figure environment for labels and captions.

**Source files covered:**
- `jmlr_paper_main.tex` — main paper (5 figures)
- `supp_routing_improvements.tex` — Supplementary A (0 `\\includegraphics`)
- `supp_benchmark_report.tex` — Supplementary B (0 `\\includegraphics`; 13 figures listed in inventory)

**Steps:**
1. Extract all `\\includegraphics` paths from all three files
2. Detect `\\fbox` placeholder figures
3. Check image files — primary search in `hypatiax/data/results/figures/`, then inventory paths
4. Full figure environment audit
5. figures **not produced by runners** (hand-crafted / architecture diagrams)
   - 5a-debug. Environment & path diagnostics (read-only)
   - 5b. Pre-audit recovery — ensure `figures/` and `tables/` are populated
   - 5c. **Suspicious / duplicate filename detector** (read-only) — flags case
     variants, typos, stray `_vN` copies, and untracked figure families that
     Steps 3-5 can't see, since they only check whether *a* file with a given
     stem exists, not whether multiple inconsistently-named files claim to be
     the same figure. Pairs with the standalone `fix_figure_names.py` script,
     which performs the actual renames/quarantine after you review this report.
   - 5d. **Auto-explain untracked figure families** (read-only) — for every
     untracked family Step 5c finds, searches the `.tex` sources for the
     family's topic and reports whether it's genuinely unmentioned (orphaned),
     discussed only in prose/a table (not a missing figure), or sitting inside
     a figure block with no image (a real gap worth adding to
     `FIGURES_INVENTORY`).
6. Copy figures-inventory files → `$ROOT/figures/`
7. Fix recipe summary

## Step 0 — Configuration

In [ ]:
import re
import shutil
from pathlib import Path

# ── Source files ───────────────────────────────────────────────────────────────
TEX_FILES = [
    "jmlr_paper_main.tex",
    "supp_routing_improvements.tex",
    "supp_benchmark_report.tex",
]

# Root of the repository / project tree
# Adjust if running from a subdirectory, e.g. ROOT = Path("..") or Path("/path/to/repo")
ROOT = Path("..")

# ── Primary search: experiment runner output ───────────────────────────────────
# Repository layout from supp_routing_improvements.tex §Reproducibility:
#   papers/2025-JMLR/hypatiax/data/results/
RUNNER_FIGURES_DIR = ROOT / "hypatiax" / "data" / "results" / "figures"

# ── figures inventory: actual on-disk paths ───────────────────────────────────────
# Inventory maps each stem to its real location in the repo.
# Runner-produced figures live in hypatiax/data/results/figures/.
# Hand-crafted figures (hypatiax_three_systems, routing_cascade_v2) land in
# figures/ (lowercase) after ci_postprocess figures_deploy copies them there.
# Cosmetic figures (fig09, fig18, fig1_seed_sweep) are in figures/ (repo root).
    # generate_figures.py now writes both .pdf and .png; inventory uses .pdf (JMLR preference).
# Supp-B sweep figures are not yet generated; paths point to where they will land
# after: scripts/generate_figures.py --experiment suppB / suppB_sc
FIGURES_INVENTORY = {
    # ── Main paper (jmlr_paper_main.tex) ──────────────────────────
    # fig:architecture §7.1 — currently \fbox placeholder
    "hypatiax_three_systems": ROOT / "figures" / "hypatiax_three_systems.pdf",
    # fig:routing_cascade §7.4 — hand-crafted; copied to figures/ by figures_deploy
    "hypatiax_algorithm1_routing_cascade_v2": ROOT / "figures" / "hypatiax_algorithm1_routing_cascade_v2.pdf",
    # fig:r2_heatmap_clipped §10.2 — cosmetic heatmap
    "fig18_r2_heatmap_improved": ROOT / "figures" / "fig18_r2_heatmap_improved.pdf",
    # fig:r2_heatmap_raw §10.2 — raw heatmap
    "fig09_r2_heatmap_regimes": ROOT / "figures" / "fig09_r2_heatmap_regimes.pdf",
    # fig:portfolio_seed_sweep §10.5 — seed sweep bar chart
    "fig1_seed_sweep": ROOT / "figures" / "fig1_seed_sweep.pdf",

    # ── Supplementary B (supp_benchmark_report.tex) — full figure inventory ─
    # All 13 figures listed in Table A.1 of supp_benchmark_report.tex.
    # Generated by scripts/generate_figures.py --experiment suppB / suppB_sc
    # from the noise-sweep and sample-complexity result JSONs.
    "fig1_r2_vs_noise":        ROOT / "figures" / "fig1_r2_vs_noise.pdf",
    "fig2_rmse_vs_noise":      ROOT / "figures" / "fig2_rmse_vs_noise.pdf",
    "fig3_time_vs_noise":      ROOT / "figures" / "fig3_time_vs_noise.pdf",
    "fig4_r2_vs_n":            ROOT / "figures" / "fig4_r2_vs_n.pdf",
    "fig5_rmse_vs_n":          ROOT / "figures" / "fig5_rmse_vs_n.pdf",
    "fig6_time_vs_n":          ROOT / "figures" / "fig6_time_vs_n.pdf",
    "fig7_recovery_vs_noise":  ROOT / "figures" / "fig7_recovery_vs_noise.pdf",
    "fig8_recovery_vs_n":      ROOT / "figures" / "fig8_recovery_vs_n.pdf",
    "fig9_minr2_vs_noise":     ROOT / "figures" / "fig9_minr2_vs_noise.pdf",
    "fig10_r2_boxplot_noise":  ROOT / "figures" / "fig10_r2_boxplot_noise.pdf",
    "fig11_recovery_heatmap":  ROOT / "figures" / "fig11_recovery_heatmap.pdf",
    "fig_runtime_comparison":  ROOT / "figures" / "fig_runtime_comparison.pdf",
    "fig_comparative_table":   ROOT / "figures" / "fig_comparative_table.pdf",
}

# ── Destination that LaTeX sees via \graphicspath{{figures/}{../figures/}} ─────
DEST_DIR = ROOT / "figures"

# ── Fallback search directories ────────────────────────────────────────────────
FALLBACK_DIRS = [
    ROOT / "figures",   # all figures land here (runner output + figures_deploy)
    ROOT,
]

EXTENSIONS = [".pdf", ".png", ".jpg", ".eps", ".svg"]

# ── Load all source files ──────────────────────────────────────────────────────
sources = {}
for tex in TEX_FILES:
    p = Path(tex)
    if p.exists():
        sources[tex] = p.read_text(encoding="utf-8")
        print(f"  Loaded  {tex}  ({len(sources[tex].splitlines())} lines)")
    else:
        print(f"  MISSING {tex}  — not found on disk")

combined_source = "\n".join(sources.values())

print()
print(f"Runner figures dir  : {RUNNER_FIGURES_DIR.resolve()}")
print(f"Destination dir     : {DEST_DIR.resolve()}")
print(f"Inventory entries   : {len(FIGURES_INVENTORY)}")

## Step 1 — Extract all \\includegraphics paths

In [ ]:
INCL_RE = re.compile(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}')

# Per-file breakdown
per_file_refs = {}
for tex, src in sources.items():
    refs = INCL_RE.findall(src)
    per_file_refs[tex] = refs

all_refs = [r for refs in per_file_refs.values() for r in refs]
unique_refs = sorted(set(all_refs))

print(f"Total \\includegraphics calls across all files: {len(all_refs)}")
print(f"Unique stems: {len(unique_refs)}")
print()

for tex, refs in per_file_refs.items():
    print(f"  {tex}  ({len(refs)} call{'s' if len(refs)!=1 else ''})")
    for r in refs:
        print(f"    {r}")
    if not refs:
        print(f"    (none)")
    print()

# Note: supp_benchmark_report.tex lists 13 figures in its Table A.1 figure inventory
# but does NOT use \includegraphics (figures are referenced only in the inventory table).
# Those stems are still tracked in FIGURES_INVENTORY and checked in Steps 3/6.
print("NOTE: supp_benchmark_report.tex lists 13 sweep figures in its Table A.1")
print("      inventory but does not embed them via \\includegraphics.")
print("      They are tracked in FIGURES_INVENTORY and are (re)generated by")
print("      scripts/generate_figures.py --experiment suppB / suppB_sc")
print("      (run via ci_postprocess.yml figures_deploy).")

## Step 2 — Detect \\fbox placeholder figures

In [ ]:
print("\\fbox occurrences across all source files:")
print("=" * 70)

total_fbox = 0
for tex, src in sources.items():
    lines = src.splitlines()
    fbox_lines = [(i+1, ln.strip()) for i, ln in enumerate(lines) if r'\fbox' in ln]
    total_fbox += len(fbox_lines)
    if fbox_lines:
        print(f"\n  {tex}  — {len(fbox_lines)} occurrence(s):")
        for lno, ctx in fbox_lines:
            print(f"    line {lno:4d}: {ctx[:110]}")

print(f"\nTotal \\fbox occurrences: {total_fbox}")
print()
print("ANALYSIS:")
print("  jmlr_paper_main.tex line 113 — \\fbox is the runtime-correction")
print("  notice box (text, not a figure placeholder). The architecture figure")
print("  (hypatiax_three_systems) is already using \\includegraphics but the image")
print("  file is MISSING on disk — see Step 3.")

## Step 3 — Check required image files

Search priority:
1. `hypatiax/data/results/figures/` — runner output
2. `FIGURES_INVENTORY` paths — hand-crafted / cosmetic figures
3. Fallback directories (`figures/`, root)

All five main-paper stems **plus** the 13 Supplementary-B sweep figures are checked.
A **MISSING FIGURES SUMMARY** is printed at the end.

In [ ]:
def find_image(stem: str):
    """Return (found, path, source_label) for a figure stem."""
    # 1. Runner output
    for ext in EXTENSIONS:
        p = RUNNER_FIGURES_DIR / (stem + ext)
        if p.exists():
            return True, p, "runner"
    # 2. Inventory path
    inv_path = FIGURES_INVENTORY.get(stem)
    if inv_path and inv_path.exists():
        return True, inv_path, "inventory"
    # 3. Fallback directories
    for d in FALLBACK_DIRS:
        for ext in EXTENSIONS:
            p = d / (stem + ext)
            if p.exists():
                return True, p, "fallback"
    return False, None, "—"


# Build full required-image registry:
#   (a) stems actually embedded via \includegraphics in any tex file
#   (b) stems listed only in FIGURES_INVENTORY (e.g. Supplementary B figures)
EMBEDDED_STEMS = set(all_refs)          # from Step 1
INVENTORY_STEMS = set(FIGURES_INVENTORY.keys())
ALL_STEMS = EMBEDDED_STEMS | INVENTORY_STEMS

STEM_DESCRIPTIONS = {
    # Main paper
    "hypatiax_three_systems":               "fig:architecture §7.1 — architecture diagram  [EMBEDDED, \\fbox placeholder]",
    "hypatiax_algorithm1_routing_cascade_v2": "fig:routing_cascade §7.4 — algorithm flow    [EMBEDDED]",
    "fig18_r2_heatmap_improved":             "fig:r2_heatmap_clipped §10.2 — clipped heatmap [EMBEDDED]",
    "fig09_r2_heatmap_regimes":              "fig:r2_heatmap_raw §10.2 — raw heatmap         [EMBEDDED]",
    "fig1_seed_sweep":                       "fig:portfolio_seed_sweep §10.5 — seed sweep      [EMBEDDED]",
    # Supplementary B sweep figures (inventory only, not \includegraphics)
    "fig1_r2_vs_noise":        "Supp-B Fig1 — Median R² vs σ               [INVENTORY]",
    "fig2_rmse_vs_noise":      "Supp-B Fig2 — Median RMSE vs σ             [INVENTORY]",
    "fig3_time_vs_noise":      "Supp-B Fig3 — Avg time vs σ                [INVENTORY]",
    "fig4_r2_vs_n":            "Supp-B Fig4 — Median R² vs n               [INVENTORY]",
    "fig5_rmse_vs_n":          "Supp-B Fig5 — Median RMSE vs n             [INVENTORY]",
    "fig6_time_vs_n":          "Supp-B Fig6 — Median time vs n             [INVENTORY]",
    "fig7_recovery_vs_noise":  "Supp-B Fig7 — Recovery rate vs σ           [INVENTORY]",
    "fig8_recovery_vs_n":      "Supp-B Fig8 — Recovery rate vs n           [INVENTORY]",
    "fig9_minr2_vs_noise":     "Supp-B Fig9 — Min R² vs σ                  [INVENTORY]",
    "fig10_r2_boxplot_noise":  "Supp-B Fig10 — Per-eq R² box plots         [INVENTORY]",
    "fig11_recovery_heatmap":  "Supp-B Fig11 — Recovery heatmap σ×n        [INVENTORY]",
    "fig_runtime_comparison":  "Supp-B — Runtime bar chart, 6 methods      [INVENTORY]",
    "fig_comparative_table":   "Supp-B — Domain×method comparison table    [INVENTORY]",
}

missing_figures = []
found_figures   = []

print("Image file availability check:")
print("=" * 95)
for stem in sorted(ALL_STEMS):
    desc = STEM_DESCRIPTIONS.get(stem, f"[unknown stem — not in descriptions]")
    found, path, src = find_image(stem)
    if found:
        print(f"  [OK]       {stem}")
        print(f"             {desc}")
        print(f"             Source : {src}  →  {path}")
        found_figures.append(stem)
    else:
        inv_path = FIGURES_INVENTORY.get(stem)
        print(f"  [MISSING]  {stem}")
        print(f"             {desc}")
        print(f"             Runner : {RUNNER_FIGURES_DIR / (stem + '.pdf')}")
        print(f"             Inventory: {inv_path if inv_path else '(not in inventory)'}")
        missing_figures.append((stem, desc))
    print()

print("=" * 95)
print(f"MISSING FIGURES SUMMARY  ({len(missing_figures)} of {len(ALL_STEMS)} total):")
print()
if missing_figures:
    embedded_missing = [(s,d) for s,d in missing_figures if s in EMBEDDED_STEMS]
    inventory_missing = [(s,d) for s,d in missing_figures if s not in EMBEDDED_STEMS]
    if embedded_missing:
        print(f"  ✗ EMBEDDED figures missing from disk ({len(embedded_missing)}) — will break LaTeX build:")
        for stem, desc in embedded_missing:
            print(f"      {stem}")
            print(f"        {desc}")
        print()
    if inventory_missing:
        print(f"  ✗ Inventory-only figures missing ({len(inventory_missing)}) — needed for Supplementary B:")
        for stem, desc in inventory_missing:
            print(f"      {stem}")
else:
    print("  ✓ All figures accounted for.")

## Step 4 — Full figure environment audit

Audits every `\\begin{figure}` … `\\end{figure}` block across all three tex files.

In [ ]:
FIG_ENV_RE = re.compile(
    r'\\begin\{figure\*?\}(.*?)\\end\{figure\*?\}',
    re.DOTALL
)

total_figs = 0
for tex, src in sources.items():
    fig_blocks = FIG_ENV_RE.findall(src)
    if not fig_blocks:
        continue
    print(f"\n{'='*80}")
    print(f"  {tex}  —  {len(fig_blocks)} figure environment(s)")
    print(f"{'='*80}")
    for i, block in enumerate(fig_blocks):
        label_m = re.search(r'\\label\{([^}]+)\}', block)
        cap_m   = re.search(r'\\caption\{([^}]{0,80})', block)
        incl_m  = re.search(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}', block)
        fbox_m  = r'\fbox' in block

        label = label_m.group(1) if label_m else "[NO LABEL]"
        cap   = cap_m.group(1).strip()[:72] if cap_m else "[NO CAPTION]"
        img   = incl_m.group(1) if incl_m else ("[fbox PLACEHOLDER]" if fbox_m else "[NO IMAGE]")

        # Determine status
        if not incl_m and not fbox_m:
            flag = "WARNING — NO IMAGE"
        elif not incl_m and fbox_m:
            flag = "WARNING — fbox PLACEHOLDER"
        else:
            found, _, _ = find_image(img)
            flag = "OK" if found else "WARNING — FILE MISSING"

        print(f"  Fig {total_figs+i+1:2d}  [{flag}]")
        print(f"          label  : {label}")
        print(f"          image  : {img}")
        print(f"          caption: {cap}…")
        print()
    total_figs += len(fig_blocks)

print(f"\nTotal figure environments across all files: {total_figs}")

## Step 5 — figures NOT produced by runners

Cross-references each required stem against `RUNNER_FIGURES_DIR`.
Anything absent from the runner output must be placed manually
(architecture diagrams, cosmetic edits, sweep plots from JSON).

figures not produced by an experiment runner fall into two groups:
- Hand-crafted, no generator: `hypatiax_three_systems`, `hypatiax_algorithm1_routing_cascade_v2`
  — must be placed in `figures/` manually.
- Script-generated: `fig09_r2_heatmap_regimes`, `fig18_r2_heatmap_improved`, `fig1_seed_sweep`
  (from `--experiment exp1`) and the 13 Supp-B sweep figures
  (from `--experiment suppB` / `suppB_sc`) are regenerated via:
```bash
python scripts/generate_figures.py --experiment exp1     --results-dir <exp1 results>     --figures-dir <exp1 results>/figures     --source auto
python scripts/generate_figures.py --experiment suppB    --results-dir <suppB results>    --figures-dir <suppB results>/figures    --source auto
python scripts/generate_figures.py --experiment suppB_sc --results-dir <suppB_sc results> --figures-dir <suppB_sc results>/figures --source auto
```
(this is exactly what `ci_postprocess.yml`'s `figures_deploy` job does before
copying everything into `figures/`).

In [ ]:
print("figures NOT produced by experiment runners")
print("(must be placed manually or regenerated from JSON)")
print("=" * 80)

non_runner = []
runner_produced = []

for stem in sorted(ALL_STEMS):
    runner_has = any(
        (RUNNER_FIGURES_DIR / (stem + ext)).exists()
        for ext in EXTENSIONS
    )
    if runner_has:
        runner_produced.append(stem)
    else:
        inv_path = FIGURES_INVENTORY.get(stem)
        if inv_path is not None and inv_path.exists():
            status = f"inventory → {inv_path}"
        elif inv_path is not None:
            status = f"MISSING — inventory path does not exist: {inv_path}"
        else:
            status = "MISSING — not in runner OR inventory"
        non_runner.append((stem, status))

# Categorise non-runner figures
arch_figs   = [s for s,_ in non_runner if "architecture" in s or "routing_cascade" in s or "three_systems" in s]
heatmap_figs= [s for s,_ in non_runner if "heatmap" in s or "r2_heatmap" in s]
sweep_figs  = [s for s,_ in non_runner if s.startswith("fig") and s not in arch_figs + heatmap_figs]

categories = [
    ("Architecture / algorithm diagrams (hand-crafted, no generator)", arch_figs),
    ("Cosmetic heatmaps (generated by scripts/generate_figures.py --experiment exp1)", heatmap_figs),
    ("Sweep & comparative plots (generated by scripts/generate_figures.py --experiment suppB / suppB_sc)", sweep_figs),
]

for cat_name, stems in categories:
    matching = [(s, st) for s, st in non_runner if s in stems]
    if not matching:
        continue
    print(f"\n  [{cat_name}]")
    for stem, status in matching:
        print(f"    {stem}")
        print(f"      → {status}")

print()
print(f"Non-runner total : {len(non_runner)} of {len(ALL_STEMS)}")
if runner_produced:
    print(f"Runner-produced  : {len(runner_produced)} — {runner_produced}")

# Regeneration commands
print()
print("Regeneration commands (run from repo root, or via ci_postprocess figures_deploy):")
print("  Cosmetic (fig09, fig18, fig1_seed_sweep):")
print("    python scripts/generate_figures.py --experiment exp1 --results-dir <exp1 results> \\")
print("      --figures-dir <exp1 results>/figures --source auto")
print("  Supplementary-B sweep figures (13 stems):")
print("    python scripts/generate_figures.py --experiment suppB --results-dir <suppB results> \\")
print("      --figures-dir <suppB results>/figures --source auto")
print("    python scripts/generate_figures.py --experiment suppB_sc --results-dir <suppB_sc results> \\")
print("      --figures-dir <suppB_sc results>/figures --source auto")

## Step 5a-debug — Environment & path diagnostics

Run **before** the Step 5b recovery guard. Prints the kernel's actual cwd,
where `ROOT` resolves to, and the real contents of every directory the
recovery/search logic depends on — at the notebook's own level *and* one
level up — so a CI failure shows exactly which directory is empty/missing
and why, instead of just the generic "both empty or missing" message.

Purely diagnostic: never raises, never modifies anything on disk.

In [ ]:
# ── Step 5a-debug: environment & path diagnostics (read-only, never raises) ──
import os
import sys
from pathlib import Path

def _ls(p: Path, max_items: int = 25):
    """Best-effort directory listing; never throws."""
    try:
        if not p.exists():
            return "<does not exist>"
        if not p.is_dir():
            return "<exists but is not a directory>"
        items = sorted(os.listdir(p))
        if not items:
            return "<exists, empty>"
        shown = items[:max_items]
        suffix = f"  (+{len(items) - max_items} more)" if len(items) > max_items else ""
        return ", ".join(shown) + suffix
    except Exception as e:
        return f"<error listing: {e!r}>"

def _tree(start: Path, depth: int = 1, prefix: str = ""):
    """Best-effort shallow directory tree; never throws."""
    if depth < 0:
        return
    try:
        entries = sorted(start.iterdir(), key=lambda p: (not p.is_dir(), p.name))
    except Exception as e:
        print(f"{prefix}<error: {e!r}>")
        return
    for entry in entries:
        marker = "/" if entry.is_dir() else ""
        print(f"{prefix}{entry.name}{marker}")
        if entry.is_dir() and depth > 0:
            _tree(entry, depth - 1, prefix + "    ")

def print_path_diagnostics(header="STEP 5a-debug — environment & path diagnostics"):
    """Print cwd, ROOT resolution, and every candidate dir Step 5b depends on,
    both at ROOT and one level up (PARENT). Read-only; never raises — safe to
    call proactively (Step 5a-debug) AND reactively right before a RuntimeError
    in Step 5b, so the diagnostics are guaranteed to land in the CI log even
    if nbconvert never writes the executed notebook back to disk.
    """
    print("=" * 80)
    print(header)
    print("=" * 80)

    print(f"\n[cwd]")
    print(f"  os.getcwd()        = {os.getcwd()}")
    print(f"  Path('.').resolve()= {Path('.').resolve()}")
    print(f"  sys.argv[0]        = {sys.argv[0] if sys.argv else '<empty>'}")

    print(f"\n[ROOT as configured in Step 0]")
    print(f"  ROOT (raw)         = {ROOT!r}")
    print(f"  ROOT.resolve()     = {ROOT.resolve()}")

    candidates = {
        "ROOT/figures":                        ROOT / "figures",
        "ROOT/tables":                         ROOT / "tables",
        # FIX-NB05-2: was a duplicate 'ROOT/figures' key (silently dropped by Python).
        # Now checks ROOT/Figures (capital-F) — the old NB05_FIGURES_DIR value that was
        # corrected in ci_postprocess; this entry should show exists=False after the fix.
        "ROOT/Figures (old NB05_FIGURES_DIR)": ROOT / "Figures",
        "ROOT/hypatiax/data/results":          ROOT / "hypatiax" / "data" / "results",
        "ROOT/hypatiax/data/results/figures":  ROOT / "hypatiax" / "data" / "results" / "figures",
    }

    # One level UP from ROOT — catches the classic "nbconvert's kernel cwd is
    # the notebook's own directory, but figures/tables were deposited one
    # level up at the repo root" mismatch.
    PARENT = ROOT.resolve().parent
    candidates_parent = {
        "PARENT/figures":                       PARENT / "figures",
        "PARENT/tables":                        PARENT / "tables",
        # FIX-NB05-2 (parent variant): was a duplicate 'PARENT/figures' key.
        "PARENT/Figures (old NB05_FIGURES_DIR)": PARENT / "Figures",
        "PARENT/hypatiax/data/results":         PARENT / "hypatiax" / "data" / "results",
        "PARENT/hypatiax/data/results/figures": PARENT / "hypatiax" / "data" / "results" / "figures",
        "PARENT/notebooks":                     PARENT / "notebooks",
    }

    print(f"\n[Candidates relative to ROOT = {ROOT.resolve()}]")
    for label, path in candidates.items():
        exists = path.exists()
        kind = "dir" if path.is_dir() else ("file" if path.is_file() else "-")
        print(f"  {label:38s} exists={exists!s:5s} kind={kind:4s} -> {_ls(path)}")

    print(f"\n[Same candidates one level up, PARENT = {PARENT}]")
    print( "  (catches: nbconvert's kernel cwd == notebook's own dir, while CI")
    print( "   populated figures/ or tables/ one level up at the repo root)")
    for label, path in candidates_parent.items():
        exists = path.exists()
        kind = "dir" if path.is_dir() else ("file" if path.is_file() else "-")
        print(f"  {label:38s} exists={exists!s:5s} kind={kind:4s} -> {_ls(path)}")

    print(f"\n[Directory tree around cwd, depth 1]")
    _tree(Path(".").resolve(), depth=1)

    print("\n" + "=" * 80)
    print("END " + header)
    print("=" * 80)

# Run once now, proactively, before Step 5b's recovery guard.
print_path_diagnostics()

## Step 5b — Pre-audit recovery: verify `figures/` and `tables/` are populated

Before the audit (Steps 6–7), checks that `repo_root/figures/` and
`repo_root/tables/` each exist and are non-empty.

If either directory is absent or empty, calls `repopulate_from_results()` to
gather figures/tables from **all** known source locations and copy them into
the repo-root flat directories — the same recovery logic that
`ci_paper_audit.yml` Job 0b performs:

- `ROOT/figures/` — hand-crafted figures with no generator, plus
  any runner-generated figures already synced here by `ci_postprocess.yml`
  (`figures_deploy`). This is the primary source per `FALLBACK_DIRS` /
  `FIGURES_INVENTORY` in the config cell.
- `hypatiax/data/results/` — raw runner output (`.pdf`/`.png` figures,
  `.tex` tables), scanned recursively as a secondary source for cases where
  `figures_deploy` hasn't run yet.

If a directory is still empty after recovery, a `RuntimeError` is raised so the
notebook stops immediately rather than producing a misleading audit.
Steps 6–7 never audit `hypatiax/data/results/` directly.

In [ ]:
# ── Step 5b: Pre-audit recovery — ensure repo_root/figures/ and repo_root/tables/ are populated ──
import shutil

FIG_DIR = ROOT / "figures"
TAB_DIR = ROOT / "tables"

def repopulate_from_results():
    """Copy figures (pdf/png) into repo_root/figures/ and tables (.tex) into
    repo_root/tables/, pulling from every known source location:
      1. ROOT/figures/          - hand-crafted figures (no generator) + anything
                                   already synced here by ci_postprocess.yml
                                   figures_deploy. Primary source.
      2. hypatiax/data/results/ - raw runner output, scanned recursively.
                                   Secondary source for results not yet synced.
    Mirrors what ci_postprocess.yml 'Copy figures & tables' and
    ci_paper_audit Job 0b perform.
    """
    OUT_BASE = ROOT / "hypatiax" / "data" / "results"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    TAB_DIR.mkdir(parents=True, exist_ok=True)

    fig_count = tab_count = 0

    # ── Source 1: RUNNER_FIGURES_DIR (hypatiax/data/results/figures/) ──────
    # FIX-NB05-3: was "ROOT/figures/" which equals FIG_DIR — a self-copy that
    # is always a no-op (dest.exists() == True because src IS dest), so
    # fig_count was always 0 and recovery silently did nothing even when figures
    # were present.  Now correctly sources from the runner output directory,
    # which is distinct from FIG_DIR and contains the actual generated figures.
    if RUNNER_FIGURES_DIR.exists():
        for p in sorted(RUNNER_FIGURES_DIR.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix.lower() in (".pdf", ".png"):
                dest = FIG_DIR / p.name
                if not dest.exists():
                    shutil.copy2(p, dest)
                    fig_count += 1

    # ── Source 2: hypatiax/data/results/ (raw runner output) ──
    if OUT_BASE.exists():
        for p in sorted(OUT_BASE.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix.lower() in (".pdf", ".png"):
                dest = FIG_DIR / p.name
                if not dest.exists():
                    shutil.copy2(p, dest)
                    fig_count += 1
            elif p.suffix.lower() == ".tex":
                exp_dir = p.parent.parent.name
                dest = TAB_DIR / f"{exp_dir}__{p.stem}.tex"
                if not dest.exists():
                    shutil.copy2(p, dest)
                    tab_count += 1

    print(f"  repopulate_from_results: {fig_count} figure(s), {tab_count} table(s) copied")
    print(f"    sources checked: {RUNNER_FIGURES_DIR} (exists={RUNNER_FIGURES_DIR.exists()}), "
          f"{OUT_BASE} (exists={OUT_BASE.exists()})")

# FIX-NB05-4: was two independent guards that each called repopulate_from_results()
# separately — if FIG_DIR was empty it ran once, then TAB_DIR triggered a second
# full scan of hypatiax/data/results/ even though both dirs are populated in one
# call.  Now: call once if either is absent/empty.
if (not FIG_DIR.exists() or not any(FIG_DIR.iterdir())
        or not TAB_DIR.exists() or not any(TAB_DIR.iterdir())):
    repopulate_from_results()

if not any(FIG_DIR.iterdir()):
    print("\n!!! FIG_DIR empty after recovery — dumping diagnostics before raising !!!\n")
    print_path_diagnostics(header="STEP 5b-FAILURE — FIG_DIR empty, diagnostics at point of failure")
    raise RuntimeError(
        f"{FIG_DIR} is empty after recovery — "
        f"checked {ROOT / 'figures'} and {ROOT / 'hypatiax' / 'data' / 'results'}, both empty or missing. "
        "Place figures in figures/ or run ci_postprocess.yml (figures_deploy), "
        "then re-run this notebook. See STEP 5b-FAILURE diagnostics above for the "
        "actual cwd/ROOT resolution and what was really on disk."
    )

if not any(TAB_DIR.iterdir()):
    print("\n!!! TAB_DIR empty after recovery — dumping diagnostics before raising !!!\n")
    print_path_diagnostics(header="STEP 5b-FAILURE — TAB_DIR empty, diagnostics at point of failure")
    raise RuntimeError(
        f"{TAB_DIR} is empty after recovery — "
        "run ci_postprocess.yml (any experiment) or commit .tex files to tables/ directly, "
        "then re-run this notebook. See STEP 5b-FAILURE diagnostics above for the "
        "actual cwd/ROOT resolution and what was really on disk."
    )

print(f"✅  figures/ : {sum(1 for _ in FIG_DIR.iterdir())} file(s)")
print(f"✅  tables/  : {sum(1 for _ in TAB_DIR.iterdir())} file(s)")
print("Pre-audit check PASSED — proceeding to Step 6.")


## Step 5c — Suspicious / duplicate filename detector

The checks above (Steps 3–5) only ask **"does *a* file matching this stem
exist?"** — they're blind to the case where *multiple, inconsistently named
files* all claim to be the same figure. That's the actual state of both
`figures/` and `hypatiax/data/results/figures/` right now:

- **Case variants** — `hypatiaX_three_systems.png` (capital `X`) alongside
  the canonical `hypatiax_three_systems.png`/`.pdf`.
- **A typo, not a version** — `hypatiax_algorithm1_rout`**`ine`**`_cascade_v2.png`
  vs. the canonical `rout`**`ing`**`_cascade_v2`. This is easy to misread as
  an intentional variant because it also carries a `_v2` suffix.
- **Untracked `_v2` figures** — `hypatiaX_three_systems_v2.png`,
  `hypatiax_instability_histogram_v2.png`, `hypatiax_instability_scatter_v2.png`.
- **An entire untracked figure family** — `hypatiax_instability_scatter`,
  `hypatiax_instability_histogram`, `hypatiax_instability_per_case` (plus their
  `.pdf`/`.png`/`_v2` copies) appear in both directories but are in **none**
  of `EMBEDDED_STEMS` or `FIGURES_INVENTORY`. Either the paper needs them and
  the inventory is out of date, or they're orphaned experiment output.

This step clusters every file in `figures/` and `hypatiax/data/results/figures/`
onto a canonical stem (correcting known case/typo variants), flags anything
that isn't an exact canonical-name match, and content-hashes files within each
cluster so you know whether the variants are byte-identical duplicates
(safe to delete) or have actually diverged (need a human to pick a winner).

This step is **read-only** — it only reports. Use the companion script
`fix_figure_names.py` (Step 6 below references it) to actually rename/quarantine.

In [ ]:
# ── Step 5c: Suspicious / duplicate filename detector (read-only) ──────────
import hashlib
import re
from collections import defaultdict

# Known, manually-verified typo fixes (filename text -> corrected text).
# Add an entry here only after confirming on disk/with the team that it's a
# typo of a canonical stem, not a deliberately different figure.
KNOWN_TYPO_FIXES = {
    "routine_cascade": "routing_cascade",
}

VERSION_SUFFIX_RE = re.compile(r'(_v\d+)$')
FIG_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".eps", ".svg"}

CANONICAL_STEMS = set(FIGURES_INVENTORY.keys()) | EMBEDDED_STEMS


def _candidate_stems(raw_stem: str):
    """All plausible canonical-stem readings of a raw filename stem: as-is,
    lowercased, with known typos fixed, and with a trailing _vN stripped —
    tried in combination, since a file can have both a case issue and a
    version suffix (e.g. hypatiaX_three_systems_v2)."""
    variants = {raw_stem, raw_stem.lower()}
    for v in list(variants):
        fixed = v
        for bad, good in KNOWN_TYPO_FIXES.items():
            fixed = fixed.replace(bad, good)
        variants.add(fixed)
    expanded = set(variants)
    for v in variants:
        m = VERSION_SUFFIX_RE.search(v)
        if m:
            expanded.add(v[: m.start()])
    return expanded


def _sha256(path, chunk_size=1 << 16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()[:12]  # short hash is enough to distinguish duplicates


def scan_directory_for_suspicious_names(directory: Path, label: str):
    """Cluster every figure file in `directory` onto a canonical stem (or an
    UNTRACKED bucket), flag case/typo/version anomalies, and hash files
    within each cluster to reveal identical-vs-diverged duplicates."""
    print(f"\n{'='*90}")
    print(f"  {label}  —  {directory.resolve()}")
    print(f"{'='*90}")

    if not directory.exists():
        print("  <directory does not exist>")
        return {}

    clusters = defaultdict(list)
    for p in sorted(directory.iterdir()):
        if not p.is_file() or p.suffix.lower() not in FIG_EXTENSIONS:
            continue
        stem = p.stem
        cands = _candidate_stems(stem)
        match = next((c for c in cands if c in CANONICAL_STEMS), None)
        key = match if match else f"UNTRACKED:{stem.lower().rstrip('0123456789').rstrip('_v')}"
        clusters[key].append(p)

    any_issue = False
    for canon in sorted(clusters):
        files = clusters[canon]
        is_untracked = canon.startswith("UNTRACKED:")
        # Flag anomalies relative to the canonical stem (skip for untracked).
        anomalies = {}
        if not is_untracked:
            for p in files:
                tags = []
                if p.stem != p.stem.lower():
                    tags.append("CASE-VARIANT")
                if VERSION_SUFFIX_RE.search(p.stem.lower()) and p.stem.lower() != canon:
                    tags.append("VERSION-SUFFIX")
                for bad in KNOWN_TYPO_FIXES:
                    if bad in p.stem.lower():
                        tags.append(f"TYPO({bad}->{KNOWN_TYPO_FIXES[bad]})")
                if tags:
                    anomalies[p] = tags

        flagged = is_untracked or bool(anomalies)
        if flagged:
            any_issue = True

        header = f"  [{'UNTRACKED FAMILY' if is_untracked else ('SUSPICIOUS' if anomalies else 'clean')}] {canon}"
        print(header)

        # Hash every file in the cluster so identical vs. diverged is explicit.
        # Compare hashes only WITHIN the same extension — a .pdf and .png of the
        # same figure are naturally different bytes; that's not a divergence,
        # it's just two renderings of the same artifact.
        hashes = {}
        for p in files:
            try:
                hashes[p] = _sha256(p)
            except Exception as e:
                hashes[p] = f"<error: {e!r}>"

        for p in files:
            tag = ""
            if p in anomalies:
                tag = f"  <-- {', '.join(anomalies[p])}"
            print(f"      {p.name:55s} sha256[:12]={hashes[p]}{tag}")

        if not is_untracked and len(files) > 1:
            by_ext = defaultdict(set)
            for p in files:
                by_ext[p.suffix.lower()].add(hashes[p])
            diverged_exts = {ext for ext, hs in by_ext.items() if len(hs) > 1}
            if diverged_exts:
                print(f"      => DIVERGED within extension(s) {sorted(diverged_exts)} — "
                      f"multiple files with the SAME extension have DIFFERENT content. "
                      f"A human must pick the winner before collapsing.")
            elif len(files) > len(by_ext):
                print(f"      => {len(files)} files, but only one distinct file per extension — "
                      f"likely just a case/typo/version-suffix naming duplicate, not a real divergence. "
                      f"Safe to collapse to canonical names per extension.")
            else:
                print(f"      => One file per extension, no duplicates within an extension.")
        print()

    if not any_issue:
        print("  ✓ No suspicious names or untracked families detected in this directory.")

    return clusters


figures_clusters = scan_directory_for_suspicious_names(DEST_DIR, "figures/ (repo root)")
runner_clusters  = scan_directory_for_suspicious_names(RUNNER_FIGURES_DIR, "hypatiax/data/results/figures/ (runner output)")

print(f"\n{'='*90}")
print("SUMMARY — Step 5c")
print(f"{'='*90}")
print("If any cluster above is flagged SUSPICIOUS or UNTRACKED FAMILY, do not")
print("hand-fix filenames in place. Instead run, from the repo root:")
print()
print("    python fix_figure_names.py            # dry run — prints the plan only")
print("    python fix_figure_names.py --apply     # actually renames + quarantines")
print()
print("See fix_figure_names.py's module docstring for exactly what it does and")
print("does not do (it never deletes a file outright — divergent duplicates are")
print("moved to figures/_superseded/, never removed).")

## Step 5d — Auto-explain untracked figure families

Step 5c finds untracked figure *files* on disk; this step answers the
question that naturally follows: **does the paper text even talk about this
topic, and if so, as a figure or as something else (prose/table)?**

For every `UNTRACKED:` cluster Step 5c found (in either `figures/` or
`hypatiax/data/results/figures/`), this step:

1. Derives a search keyword from the stem (e.g. `hypatiax_instability_scatter`
   → `instability`), stripping generic prefixes (`hypatiax_`, `fig_`, `figNN_`)
   and plot-type words (`scatter`, `histogram`, `heatmap`, `boxplot`, `per_case`, …)
   that would make the search too broad to be useful.
2. Searches all three `.tex` sources for that keyword.
3. Classifies what it finds into one of:
   - **NOT_MENTIONED** — the keyword appears nowhere. Likely an orphaned /
     leftover artifact from an earlier draft or exploratory analysis.
     Candidate for moving out of `figures/` entirely.
   - **PROSE_AND_TABLE_ONLY** — the keyword is discussed, but only in prose
     and/or inside a `\begin{table}…\end{table}` block, never inside a
     `\begin{figure}` block or an `\includegraphics` call. This means the
     author's editorial choice was to present the result as text/a table —
     **not** a missing figure. The file(s) on disk are most likely the
     exploratory plots used to *derive* the numbers in that table.
   - **FIGURE_ENV_NO_IMAGE** — the keyword appears inside a `\begin{figure}`
     block, but that block has no `\includegraphics` (might be an `\fbox`
     placeholder, or a figure stub waiting on content). **Likely a real gap**
     — the stem should probably be added to `FIGURES_INVENTORY`.
   - **FIGURE_ENV_FOUND** — the keyword appears inside a `\begin{figure}`
     block that *does* have an `\includegraphics`, but the image stem inside
     doesn't match anything in this untracked family. Worth a manual look —
     either a separate figure happens to share the keyword, or the family's
     naming has drifted further than Step 5c's matching can bridge.

This step is **read-only** and purely diagnostic — it never edits
`FIGURES_INVENTORY`, the `.tex` files, or anything on disk. It exists so the
"is this orphaned or missing?" question Step 5c can't answer gets answered
automatically instead of requiring a manual `grep` every time.

In [ ]:
# ── Step 5d: Auto-explain untracked figure families (read-only) ───────────
import re as _re5d  # local alias, avoids clobbering the `re` imported in Step 5c

FIG_ENV_RE_5D   = _re5d.compile(r'\\begin\{figure\*?\}(.*?)\\end\{figure\*?\}', _re5d.DOTALL)
TABLE_ENV_RE_5D = _re5d.compile(r'\\begin\{table\}(.*?)\\end\{table\}', _re5d.DOTALL)
INCL_RE_5D      = _re5d.compile(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}')

# Generic prefixes/suffixes/plot-type words stripped when deriving a search
# keyword from a stem, so "hypatiax_instability_scatter" and
# "hypatiax_instability_histogram" both collapse to the same keyword
# ("instability") instead of being treated as unrelated topics.
GENERIC_PREFIX_RE = _re5d.compile(r'^(hypatiax_|fig\d*_|fig_)')
GENERIC_VERSION_RE = _re5d.compile(r'_v\d+$')
GENERIC_PLOT_WORDS = {
    "scatter", "histogram", "per", "case", "plot", "chart",
    "heatmap", "boxplot", "comparison", "table",
}


def derive_keyword(stem: str) -> str:
    """Collapse a figure stem down to the distinctive topic word(s) shared
    across a family of related filenames. Best-effort heuristic, not exact:
    if it picks an unhelpful keyword for a given repo, override by editing
    GENERIC_PLOT_WORDS or hardcoding an exception here."""
    s = stem.lower()
    s = GENERIC_PREFIX_RE.sub('', s)
    s = GENERIC_VERSION_RE.sub('', s)
    tokens = [t for t in s.split('_') if t and t not in GENERIC_PLOT_WORDS]
    return tokens[0] if tokens else s


def classify_keyword_presence(keyword: str, sources: dict):
    """Search every loaded .tex source for `keyword` and classify what kind
    of mention (if any) it gets: none, prose/table-only, or figure-related."""
    kw_re = _re5d.compile(_re5d.escape(keyword), _re5d.IGNORECASE)

    total_mentions = 0
    in_figure_env = False
    in_table_env = False
    matching_fig_images = []
    hit_files = []

    for fname, src in sources.items():
        mentions = kw_re.findall(src)
        if mentions:
            hit_files.append((fname, len(mentions)))
        total_mentions += len(mentions)

        for block in FIG_ENV_RE_5D.findall(src):
            if kw_re.search(block):
                in_figure_env = True
                matching_fig_images += INCL_RE_5D.findall(block)

        for block in TABLE_ENV_RE_5D.findall(src):
            if kw_re.search(block):
                in_table_env = True

    if total_mentions == 0:
        status = "NOT_MENTIONED"
    elif in_figure_env:
        status = "FIGURE_ENV_FOUND" if matching_fig_images else "FIGURE_ENV_NO_IMAGE"
    elif in_table_env:
        status = "PROSE_AND_TABLE_ONLY"
    else:
        status = "PROSE_ONLY"

    return {
        "status": status,
        "total_mentions": total_mentions,
        "hit_files": hit_files,
        "in_figure_env": in_figure_env,
        "in_table_env": in_table_env,
        "matching_fig_images": matching_fig_images,
    }


STATUS_EXPLANATIONS = {
    "NOT_MENTIONED":
        "Topic isn't mentioned anywhere in the .tex sources. Likely an orphaned / "
        "leftover artifact from an earlier draft or exploratory analysis run — not "
        "something the current paper draft needs. Candidate for moving out of "
        "figures/ (e.g. into an analysis/ or archive/ folder) so it stops appearing "
        "in this audit.",
    "PROSE_ONLY":
        "Topic is discussed in prose but never inside a figure or table environment. "
        "Likely just narrative text — no figure is implied.",
    "PROSE_AND_TABLE_ONLY":
        "Topic is discussed in prose AND presented via a \\begin{table}, but never "
        "inside a \\begin{figure} or \\includegraphics. This looks like a deliberate "
        "editorial choice to present the result as text/a table rather than a "
        "figure — NOT a missing figure. The on-disk file(s) are most likely the "
        "exploratory plots used to derive the table's numbers, kept around as "
        "working artifacts rather than paper deliverables.",
    "FIGURE_ENV_NO_IMAGE":
        "Topic appears inside a \\begin{figure}...\\end{figure} block, but that "
        "block has no \\includegraphics (possibly an \\fbox placeholder, or a "
        "stub awaiting content). This looks like a REAL GAP — consider adding "
        "this stem to FIGURES_INVENTORY in Step 0.",
    "FIGURE_ENV_FOUND":
        "Topic appears inside a figure block that already has an \\includegraphics, "
        "but the embedded image stem doesn't match this untracked family. Either "
        "a different figure happens to share the keyword, or the family's actual "
        "name has drifted further than this step's matching can bridge — take a "
        "manual look at the figure block before concluding either way.",
}


# ── Collect every UNTRACKED: family found by Step 5c, across both dirs ─────
untracked_keys = set()
for clusters in (figures_clusters, runner_clusters):
    for key in clusters:
        if key.startswith("UNTRACKED:"):
            untracked_keys.add(key)

print("=" * 90)
print("Step 5d — Auto-explain untracked figure families")
print("=" * 90)

if not untracked_keys:
    print("\n  ✓ No untracked families to explain — Step 5c found nothing outside "
          "FIGURES_INVENTORY / EMBEDDED_STEMS.")
else:
    # Group raw untracked stems by derived keyword, since several stems
    # (scatter/histogram/per_case variants) usually collapse to one topic.
    by_keyword = defaultdict(list)
    for key in sorted(untracked_keys):
        raw_stem = key.split("UNTRACKED:", 1)[1]
        by_keyword[derive_keyword(raw_stem)].append(raw_stem)

    for keyword in sorted(by_keyword):
        stems = by_keyword[keyword]
        print(f"\n  Keyword: '{keyword}'   (covers stems: {', '.join(stems)})")
        result = classify_keyword_presence(keyword, sources)
        print(f"    Status          : {result['status']}")
        print(f"    Total mentions  : {result['total_mentions']}"
              + (f"  ({', '.join(f'{f}:{n}' for f, n in result['hit_files'])})"
                 if result['hit_files'] else ""))
        if result["matching_fig_images"]:
            print(f"    Fig-env images  : {result['matching_fig_images']}")
        print(f"    => {STATUS_EXPLANATIONS[result['status']]}")

    print(f"\n{'-'*90}")
    print("This is diagnostic only — nothing was changed. If a family comes back")
    print("FIGURE_ENV_NO_IMAGE, add its stem(s) to FIGURES_INVENTORY (Step 0) and to")
    print("CANONICAL_STEMS in fix_figure_names.py, then re-run Steps 3, 5c and 6.")

## Step 6 — Copy inventory figures → `$ROOT/figures/`

Copies every figure that exists at its `FIGURES_INVENTORY` path to `DEST_DIR`
(the `figures/` directory that LaTeX reads via `\graphicspath{{figures/}{../figures/}}`).

Set `DRY_RUN = True` to preview without writing anything. Default is `False` — copies are performed on run.

In [ ]:
DRY_RUN = False   # ← set True to preview without writing

DEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Copying inventory figures → {DEST_DIR.resolve()}")
print("=" * 80)

copied, skipped, not_found = [], [], []

for stem, src_path in FIGURES_INVENTORY.items():
    if not src_path.exists():
        print(f"  [NOT FOUND]  {stem}")
        print(f"               Expected : {src_path}")
        not_found.append(stem)
        continue

    dest_path = DEST_DIR / src_path.name
    if dest_path.exists():
        print(f"  [SKIP]       {src_path.name}  (already in {DEST_DIR})")
        skipped.append(stem)
        continue

    if not DRY_RUN:
        shutil.copy2(src_path, dest_path)
        print(f"  [COPIED]     {src_path}")
        print(f"               → {dest_path}")
    else:
        print(f"  [WOULD COPY] {src_path}")
        print(f"               → {dest_path}")
    copied.append(stem)
    print()

print("=" * 80)
print(f"Copied    : {len(copied)}")
print(f"Skipped   : {len(skipped)}  (already present)")
print(f"Not found : {len(not_found)}  (source missing — see Fix recipe below)")

if DRY_RUN and (copied or not_found):
    print()
    print("Set DRY_RUN = False and re-run this cell to perform the copies.")

## Step 7 — Fix recipe

Re-checks all stems after Step 6 (in case copies were performed) and prints
only the actions still required.

In [ ]:
# Step 7 — Fix recipe (post-fix state)
print("FIX RECIPE — post-fix state")
print("=" * 80)

# Re-check all stems
still_missing = []
for stem in list(FIGURES_INVENTORY.keys()) + required_stems:
    found, path, src_lbl = find_image(stem)
    if not found:
        still_missing.append(stem)

if not still_missing:
    print("  ✓ All figures resolved. No manual action required.")
else:
    for s in still_missing:
        print(f"  STILL MISSING: {s}")

print()
print("── Additional checks ────────────────────────────────────────────────")
print("  1. \\fbox at jmlr_paper_main.tex line ~113 is a runtime-correction")
     print("     notice box (text), NOT a figure placeholder. ✅")
print("  2. supp_benchmark_report.tex references 13 figures in Table A.1 but")
print("     uses NO \\includegraphics — regenerate from JSON via:")
print("       python scripts/generate_figures.py --experiment suppB")
print("       python scripts/generate_figures.py --experiment suppB_sc")
print("  3. Citation key fixes applied to supp_benchmark_report.tex: ✅")
print("       • udrescu2020aifeynman — bibitem present (cited in supp only)")
print("       • meidani2024snip — bibitem added")
print("       • lacava2021contemporary — confirmed present in references.bib")


FIX RECIPE — post-fix state
  ✓ All figures resolved. No manual action required.

── Additional checks ────────────────────────────────────────────────
  1. \fbox at jmlr_paper_main.tex line ~113 is a runtime-correction
     notice box (text), NOT a figure placeholder. ✅
  2. supp_benchmark_report.tex references 13 figures in Table A.1 but
     uses NO \includegraphics — regenerate from JSON via:
       python scripts/generate_figures.py --experiment suppB
       python scripts/generate_figures.py --experiment suppB_sc
  3. Citation key fixes applied to supp_benchmark_report.tex: ✅
       • udrescu2020aifeynman — bibitem present (cited in supp only)
       • meidani2024snip — bibitem added
       • lacava2021contemporary — confirmed present in references.bib
